# **PBSMT**


## **1. building toy model**

## **1. Impoert libraries**

In [ ]:
!pip install sacrebleu
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 65.6 MB/s eta 0:00:00


## **Run code**

In [ ]:
# ============================================================
# FAST TOY 3-LEVEL PBSMT MODEL
# EN <-> IT
# ============================================================

# !pip install datasets sacrebleu gensim scipy

from datasets import load_dataset
import sacrebleu
import pandas as pd
import re, math, time, platform
import numpy as np
from collections import Counter, defaultdict
from gensim.models import Word2Vec
from scipy.linalg import orthogonal_procrustes


# ============================================================
# Fast settings
# ============================================================

EN_TRAIN_SPLIT = "train[1000:11000]"
IT_TRAIN_SPLIT = "train[20000:30000]"
WORD2VEC_EPOCHS = 5
IBM1_ITERS = 3
SYNTHETIC_SENTENCES = 800
BEAM_SIZE = 5
SIZES = [50, 100, 250, 500]

start_time = time.time()


# ============================================================
# Runtime information
# ============================================================

def detect_runtime():
    runtime = "CPU"

    try:
        import torch
        if torch.cuda.is_available():
            runtime = f"GPU: {torch.cuda.get_device_name(0)}"
    except:
        pass

    try:
        import torch_xla.core.xla_model as xm
        runtime = "TPU"
    except:
        pass

    return runtime

runtime_type = detect_runtime()

print("Runtime device:", runtime_type)
print("Python version:", platform.python_version())


# ============================================================
# Utilities
# ============================================================

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower(), re.UNICODE)

def detokenize(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    return text


# ============================================================
# LEVEL 1 — INITIALIZATION
# ============================================================

# Level 1.1 — Train separate embedding spaces

def train_word2vec(sentences, vector_size=80, window=5, min_count=2):
    tokenized = [tokenize(s) for s in sentences]

    return Word2Vec(
        sentences=tokenized,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=4,
        sg=1,
        epochs=WORD2VEC_EPOCHS
    )


# Level 1.2 — Align embedding spaces using matrix W

def create_seed_dictionary(src_model, tgt_model, max_pairs=700):
    src_words = set(src_model.wv.index_to_key)
    tgt_words = set(tgt_model.wv.index_to_key)

    shared_words = list(src_words.intersection(tgt_words))
    return [(w, w) for w in shared_words[:max_pairs]]


def learn_alignment_matrix_W(src_model, tgt_model, seed_pairs):
    X, Y = [], []

    for sw, tw in seed_pairs:
        if sw in src_model.wv and tw in tgt_model.wv:
            X.append(src_model.wv[sw])
            Y.append(tgt_model.wv[tw])

    X = np.array(X)
    Y = np.array(Y)

    W, _ = orthogonal_procrustes(X, Y)
    return W


# Level 1.3 — Retrieve nearest neighbors
# Level 1.4 — Build phrase table
# Level 1.5 — Convert scores to probabilities

def build_embedding_phrase_table(src_model, tgt_model, W, top_k=6):
    phrase_table = {}

    tgt_words = tgt_model.wv.index_to_key
    tgt_vectors = np.array([tgt_model.wv[w] for w in tgt_words])
    tgt_vectors = tgt_vectors / (np.linalg.norm(tgt_vectors, axis=1, keepdims=True) + 1e-9)

    for src_word in src_model.wv.index_to_key:
        src_vec = src_model.wv[src_word]
        mapped_vec = src_vec @ W
        mapped_vec = mapped_vec / (np.linalg.norm(mapped_vec) + 1e-9)

        sims = tgt_vectors @ mapped_vec
        best_ids = np.argsort(-sims)[:top_k]
        best_scores = sims[best_ids]

        exp_scores = np.exp(best_scores * 10)
        probs = exp_scores / exp_scores.sum()

        phrase_table[(src_word,)] = [
            ((tgt_words[idx],), float(prob))
            for idx, prob in zip(best_ids, probs)
        ]

    return phrase_table


# ============================================================
# LEVEL 1 EXTRA — FAST IBM1 PHRASE EXTRACTION
# ============================================================

def initialize_translation_probs(pairs, src_vocab, tgt_vocab):
    cooc = defaultdict(Counter)

    for src_sent, tgt_sent in pairs:
        src_tokens = set([w for w in tokenize(src_sent) if w in src_vocab])
        tgt_tokens = set([w for w in tokenize(tgt_sent) if w in tgt_vocab])

        for sw in src_tokens:
            for tw in tgt_tokens:
                cooc[sw][tw] += 1

    trans_probs = {}

    for sw, candidates in cooc.items():
        total = sum(candidates.values())
        trans_probs[sw] = {tw: c / total for tw, c in candidates.items()}

    return trans_probs


def train_ibm1(pairs, src_vocab, tgt_vocab, num_iters=IBM1_ITERS):
    trans_probs = initialize_translation_probs(pairs, src_vocab, tgt_vocab)

    for iteration in range(num_iters):
        print(f"IBM Model 1 iteration {iteration + 1}/{num_iters}")

        count = defaultdict(Counter)
        total_src = Counter()

        for src_sent, tgt_sent in pairs:
            src = [w for w in tokenize(src_sent) if w in src_vocab]
            tgt = [w for w in tokenize(tgt_sent) if w in tgt_vocab]

            if len(src) == 0 or len(tgt) == 0:
                continue

            for tw in tgt:
                norm = sum(trans_probs.get(sw, {}).get(tw, 1e-12) for sw in src)

                if norm == 0:
                    continue

                for sw in src:
                    delta = trans_probs.get(sw, {}).get(tw, 1e-12) / norm
                    count[sw][tw] += delta
                    total_src[sw] += delta

        for sw in count:
            trans_probs[sw] = {
                tw: count[sw][tw] / total_src[sw]
                for tw in count[sw]
            }

    return trans_probs


def get_word_alignment(src_tokens, tgt_tokens, trans_probs):
    alignment = []

    for i, sw in enumerate(src_tokens):
        if sw not in trans_probs:
            continue

        best_j = None
        best_p = 0.0

        for j, tw in enumerate(tgt_tokens):
            p = trans_probs.get(sw, {}).get(tw, 0.0)

            if p > best_p:
                best_p = p
                best_j = j

        if best_j is not None and best_p > 0:
            alignment.append((i, best_j))

    return alignment


def extract_phrase_pairs(src_tokens, tgt_tokens, alignment, max_phrase_len=3):
    phrase_pairs = []

    if not alignment:
        return phrase_pairs

    align_dict = defaultdict(list)

    for i, j in alignment:
        align_dict[i].append(j)

    for i1 in range(len(src_tokens)):
        for i2 in range(i1, min(len(src_tokens), i1 + max_phrase_len)):
            aligned_js = []

            for i in range(i1, i2 + 1):
                aligned_js.extend(align_dict.get(i, []))

            if not aligned_js:
                continue

            j1, j2 = min(aligned_js), max(aligned_js)

            if j2 - j1 + 1 > max_phrase_len:
                continue

            src_phrase = tuple(src_tokens[i1:i2+1])
            tgt_phrase = tuple(tgt_tokens[j1:j2+1])

            phrase_pairs.append((src_phrase, tgt_phrase))

    return phrase_pairs


def normalize_phrase_table(phrase_counts, top_k=6):
    phrase_table = {}

    for sp, candidates in phrase_counts.items():
        total = sum(candidates.values())
        best = candidates.most_common(top_k)

        phrase_table[sp] = [
            (tp, c / total)
            for tp, c in best
        ]

    return phrase_table


def build_ibm_phrase_table(pairs, trans_probs, max_phrase_len=3, top_k=6):
    phrase_counts = defaultdict(Counter)

    for src_sent, tgt_sent in pairs:
        src = tokenize(src_sent)
        tgt = tokenize(tgt_sent)

        if len(src) == 0 or len(tgt) == 0:
            continue

        alignment = get_word_alignment(src, tgt, trans_probs)

        phrase_pairs = extract_phrase_pairs(
            src,
            tgt,
            alignment,
            max_phrase_len=max_phrase_len
        )

        for sp, tp in phrase_pairs:
            phrase_counts[sp][tp] += 1

        for sw in src:
            if sw in trans_probs:
                best = sorted(
                    trans_probs[sw].items(),
                    key=lambda x: x[1],
                    reverse=True
                )[:top_k]

                for tw, p in best:
                    phrase_counts[(sw,)][(tw,)] += p

    return normalize_phrase_table(phrase_counts, top_k=top_k)


def merge_phrase_tables(table1, table2, top_k=6):
    counts = defaultdict(Counter)

    for table, weight in [(table1, 1.0), (table2, 2.0)]:
        for sp, candidates in table.items():
            for tp, prob in candidates:
                counts[sp][tp] += weight * prob

    return normalize_phrase_table(counts, top_k=top_k)


def build_full_phrase_table(src_sentences, tgt_sentences, src_model, tgt_model, top_k=6):
    pairs = list(zip(src_sentences, tgt_sentences))

    seed_pairs = create_seed_dictionary(src_model, tgt_model)
    W = learn_alignment_matrix_W(src_model, tgt_model, seed_pairs)

    embedding_table = build_embedding_phrase_table(
        src_model,
        tgt_model,
        W,
        top_k=top_k
    )

    src_vocab = set(src_model.wv.index_to_key)
    tgt_vocab = set(tgt_model.wv.index_to_key)

    trans_probs = train_ibm1(
        pairs,
        src_vocab,
        tgt_vocab,
        num_iters=IBM1_ITERS
    )

    ibm_table = build_ibm_phrase_table(
        pairs,
        trans_probs,
        max_phrase_len=3,
        top_k=top_k
    )

    return merge_phrase_tables(
        embedding_table,
        ibm_table,
        top_k=top_k
    )


# ============================================================
# LEVEL 2 — PBSMT TRANSLATOR
# ============================================================

# Level 2.1 — Phrase table from Level 1

# Level 2.2 — Train target-language model

def train_bigram_lm(sentences):
    unigram = Counter()
    bigram = Counter()
    vocab = set()

    for sent in sentences:
        tokens = ["<s>"] + tokenize(sent) + ["</s>"]

        vocab.update(tokens)
        unigram.update(tokens)

        for i in range(len(tokens) - 1):
            bigram[(tokens[i], tokens[i+1])] += 1

    return unigram, bigram, vocab


# Level 2.3 — LM fluency score

def lm_score(tokens, unigram, bigram, vocab):
    tokens = ["<s>"] + tokens + ["</s>"]
    V = len(vocab)
    score = 0.0

    for i in range(len(tokens) - 1):
        prob = (bigram[(tokens[i], tokens[i+1])] + 1) / (unigram[tokens[i]] + V)
        score += math.log(prob)

    return score


# Level 2.4 — Toy reordering model

def reorder_variants(segment_tokens):
    variants = [segment_tokens]

    if len(segment_tokens) == 2:
        variants.append([segment_tokens[1], segment_tokens[0]])

    if len(segment_tokens) == 3:
        variants.append([segment_tokens[1], segment_tokens[0], segment_tokens[2]])
        variants.append([segment_tokens[0], segment_tokens[2], segment_tokens[1]])

    return variants


# Level 2.5 — Decoder combines phrase score + LM + reordering

def translate_pbsmt(
    sentence,
    phrase_table,
    unigram,
    bigram,
    vocab,
    max_phrase_len=3,
    beam_size=BEAM_SIZE,
    lm_weight=0.03,
    reorder_penalty=-0.30,
    copy_penalty=-10.0
):
    src = tokenize(sentence)
    beams = [([], 0.0, 0)]

    while True:
        new_beams = []
        finished = True

        for out_tokens, phrase_score, i in beams:

            if i >= len(src):
                new_beams.append((out_tokens, phrase_score, i))
                continue

            finished = False
            matched_any = False

            for n in range(max_phrase_len, 0, -1):
                phrase = tuple(src[i:i+n])

                if phrase in phrase_table:
                    matched_any = True

                    for tgt_phrase, prob in phrase_table[phrase]:
                        base_tokens = list(tgt_phrase)

                        for variant in reorder_variants(base_tokens):
                            new_score = phrase_score + math.log(prob + 1e-12)
                            new_score += 0.25 * n

                            if variant != base_tokens:
                                new_score += reorder_penalty

                            new_beams.append((out_tokens + variant, new_score, i+n))

            if not matched_any:
                new_beams.append((out_tokens + [src[i]], phrase_score + copy_penalty, i+1))

        if finished:
            break

        scored = []

        for out_tokens, phrase_score, i in new_beams:
            total = phrase_score + lm_weight * lm_score(
                out_tokens,
                unigram,
                bigram,
                vocab
            )

            scored.append((total, out_tokens, phrase_score, i))

        scored = sorted(scored, reverse=True)[:beam_size]
        beams = [(out, ps, i) for total, out, ps, i in scored]

    return detokenize(beams[0][0])


# ============================================================
# LEVEL 3 — BIDIRECTIONAL BACK-TRANSLATION
# ============================================================

# Level 3.1 — Translate English corpus into synthetic Italian
# Level 3.2 — Translate Italian corpus into synthetic English

def create_synthetic_pairs(src_sentences, phrase_table, unigram, bigram, vocab, max_sentences):
    synthetic_pairs = []

    for src in src_sentences[:max_sentences]:
        hyp_tgt = translate_pbsmt(
            src,
            phrase_table,
            unigram,
            bigram,
            vocab
        )
        synthetic_pairs.append((src, hyp_tgt))

    return synthetic_pairs


# Level 3.3 — Update phrase table from synthetic corpus

def update_phrase_table_with_synthetic(original_phrase_table, synthetic_pairs, top_k=6):
    phrase_counts = defaultdict(Counter)

    for sp, candidates in original_phrase_table.items():
        for tp, prob in candidates:
            phrase_counts[sp][tp] += prob

    for src_sent, tgt_sent in synthetic_pairs:
        src = tokenize(src_sent)
        tgt = tokenize(tgt_sent)

        if len(src) == 0 or len(tgt) == 0:
            continue

        for i, sw in enumerate(src):
            j = round(i * (len(tgt) - 1) / max(len(src) - 1, 1))

            if 0 <= j < len(tgt):
                phrase_counts[(sw,)][(tgt[j],)] += 0.5

    return normalize_phrase_table(phrase_counts, top_k=top_k)


def bidirectional_back_translation_iteration(
    en_train,
    it_train,
    en_to_it_table,
    it_to_en_table,
    it_lm,
    en_lm,
    max_sentences=SYNTHETIC_SENTENCES
):
    it_unigram, it_bigram, it_vocab = it_lm
    en_unigram, en_bigram, en_vocab = en_lm

    print("\nLEVEL 3.1 — Creating synthetic EN→IT corpus...")
    synthetic_en_it = create_synthetic_pairs(
        en_train,
        en_to_it_table,
        it_unigram,
        it_bigram,
        it_vocab,
        max_sentences=max_sentences
    )

    print("\nLEVEL 3.2 — Creating synthetic IT→EN corpus...")
    synthetic_it_en = create_synthetic_pairs(
        it_train,
        it_to_en_table,
        en_unigram,
        en_bigram,
        en_vocab,
        max_sentences=max_sentences
    )

    print("\nLEVEL 3.3 — Updating EN→IT and IT→EN phrase tables...")

    en_to_it_updated = update_phrase_table_with_synthetic(
        en_to_it_table,
        synthetic_en_it
    )

    it_to_en_updated = update_phrase_table_with_synthetic(
        it_to_en_table,
        synthetic_it_en
    )

    return en_to_it_updated, it_to_en_updated


# ============================================================
# Evaluation
# ============================================================

def evaluate_en_to_it(phrase_table, it_unigram, it_bigram, it_vocab, sizes):
    results = []

    for n in sizes:
        print(f"\nEvaluating EN→IT on first {n} sentences...")

        test_data = load_dataset(
            "Helsinki-NLP/opus_books",
            "en-it",
            split=f"train[:{n}]"
        )

        sources = [x["translation"]["en"] for x in test_data]
        references = [x["translation"]["it"] for x in test_data]

        hypotheses = [
            translate_pbsmt(
                src,
                phrase_table,
                it_unigram,
                it_bigram,
                it_vocab
            )
            for src in sources
        ]

        bleu = sacrebleu.corpus_bleu(hypotheses, [references])

        print(f"BLEU = {bleu.score:.2f}")

        results.append({
            "Number of Sentences": n,
            "BLEU": round(bleu.score, 2)
        })

    return pd.DataFrame(results)


# ============================================================
# MAIN EXPERIMENT
# ============================================================

sizes = SIZES

# Load English and Italian monolingual corpora from different ranges.
# Therefore, the sentences are not translations of each other.
en_data = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-it",
    split=EN_TRAIN_SPLIT
)

it_data = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-it",
    split=IT_TRAIN_SPLIT
)

en_train = [x["translation"]["en"] for x in en_data]
it_train = [x["translation"]["it"] for x in it_data]


# -------------------------
# Run Level 1
# -------------------------

print("\nLEVEL 1.1 — Training separate English and Italian embeddings...")
en_model = train_word2vec(en_train)
it_model = train_word2vec(it_train)

print("\nLEVEL 1.2–1.5 — Building EN→IT phrase table...")
en_to_it_table = build_full_phrase_table(
    en_train,
    it_train,
    en_model,
    it_model
)

print("EN→IT phrase table size:", len(en_to_it_table))

print("\nLEVEL 1.2–1.5 — Building IT→EN phrase table...")
it_to_en_table = build_full_phrase_table(
    it_train,
    en_train,
    it_model,
    en_model
)

print("IT→EN phrase table size:", len(it_to_en_table))


# -------------------------
# Run Level 2
# -------------------------

print("\nLEVEL 2.1 — Phrase tables ready.")
print("LEVEL 2.2 — Training Italian and English language models...")

it_lm = train_bigram_lm(it_train)
en_lm = train_bigram_lm(en_train)

it_unigram, it_bigram, it_vocab = it_lm

print("\nLEVEL 2.3–2.5 — Evaluating initial PBSMT translator...")

before_bt = evaluate_en_to_it(
    en_to_it_table,
    it_unigram,
    it_bigram,
    it_vocab,
    sizes
)

before_bt = before_bt.rename(columns={"BLEU": "Before Bidirectional BT BLEU"})


# -------------------------
# Run Level 3
# -------------------------

en_to_it_table_bt, it_to_en_table_bt = bidirectional_back_translation_iteration(
    en_train,
    it_train,
    en_to_it_table,
    it_to_en_table,
    it_lm,
    en_lm,
    max_sentences=SYNTHETIC_SENTENCES
)

print("\nEvaluating after bidirectional back-translation...")

after_bt = evaluate_en_to_it(
    en_to_it_table_bt,
    it_unigram,
    it_bigram,
    it_vocab,
    sizes
)

after_bt = after_bt.rename(columns={"BLEU": "After Bidirectional BT BLEU"})


# -------------------------
# Final results
# -------------------------

final_table = before_bt.merge(after_bt, on="Number of Sentences")

print("\nFinal Fast Toy PBSMT 3-Level Results:")
display(final_table)


# ============================================================
# Runtime summary
# ============================================================

end_time = time.time()
elapsed = end_time - start_time

print("\nRuntime Summary")
print("----------------")
print("Device used:", runtime_type)
print(f"Total time: {elapsed:.2f} seconds")
print(f"Total time: {elapsed/60:.2f} minutes")

Runtime device: CPU
Python version: 3.12.13


README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

en-it/train-00000-of-00001.parquet:   0%|          | 0.00/5.73M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/32332 [00:00<?, ? examples/s]


LEVEL 1.1 — Training separate English and Italian embeddings...

LEVEL 1.2–1.5 — Building EN→IT phrase table...
IBM Model 1 iteration 1/3
IBM Model 1 iteration 2/3
IBM Model 1 iteration 3/3
EN→IT phrase table size: 217465

LEVEL 1.2–1.5 — Building IT→EN phrase table...
IBM Model 1 iteration 1/3
IBM Model 1 iteration 2/3
IBM Model 1 iteration 3/3
IT→EN phrase table size: 140778

LEVEL 2.1 — Phrase tables ready.
LEVEL 2.2 — Training Italian and English language models...

LEVEL 2.3–2.5 — Evaluating initial PBSMT translator...

Evaluating EN→IT on first 50 sentences...
BLEU = 0.21

Evaluating EN→IT on first 100 sentences...
BLEU = 0.18

Evaluating EN→IT on first 250 sentences...
BLEU = 0.28

Evaluating EN→IT on first 500 sentences...
BLEU = 0.23

LEVEL 3.1 — Creating synthetic EN→IT corpus...

LEVEL 3.2 — Creating synthetic IT→EN corpus...

LEVEL 3.3 — Updating EN→IT and IT→EN phrase tables...

Evaluating after bidirectional back-translation...

Evaluating EN→IT on first 50 sentences...


,Number of Sentences,Before Bidirectional BT BLEU,After Bidirectional BT BLEU
0,50,0.21,0.19
1,100,0.18,0.17
2,250,0.28,0.19
3,500,0.23,0.17



Runtime Summary
----------------
Device used: CPU
Total time: 1130.37 seconds
Total time: 18.84 minutes
